In [ ]:
import torch

# The following is vibe-coded with Gemini; fixed by Kevin Lim and Zihan Guo

# Use float64 for maximum precision with exponential terms
torch.set_default_dtype(torch.float64)

def calculate_J(R1, R2, S0, theta):

    # Slicing the parameters
    idx = 0
    nus1 = theta[idx : idx + R1]; idx += R1
    sigs1_raw = theta[idx : idx + R1]; idx += R1
    nus2 = theta[idx : idx + R2]; idx += R2
    sigs2_raw = theta[idx : idx + R2]; idx += R2

    nus1 = torch.cat([nus1, torch.tensor([S0])])
    nus2 = torch.cat([torch.tensor([S0]), nus2])

    # Enforce strict positivity on sigmas using softplus
    sigs1 = torch.nn.functional.softplus(sigs1_raw) + 1e-6
    sigs2 = torch.nn.functional.softplus(sigs2_raw) + 1e-6

    # Calculate Unit Base Coefficients (Setting lam1 = 1, lam2 = 1)

    # Left Side
    c_v1_unit = []
    dist0 = nus1[1] - nus1[0]
    c_v1_unit.append((-torch.exp(-dist0/sigs1[0]), torch.exp(dist0/sigs1[0])))

    # recursively calculating c^1
    for j in range(R1 - 1):
        P = c_v1_unit[j][0] + c_v1_unit[j][1]
        S = (1/sigs1[j]) * (-c_v1_unit[j][0] + c_v1_unit[j][1])
        dist = nus1[j+2] - nus1[j+1]
        c_v1_unit.append((0.5 * (P - sigs1[j+1]*S) * torch.exp(-dist/sigs1[j+1]),
                          0.5 * (P + sigs1[j+1]*S) * torch.exp(dist/sigs1[j+1])))

    # Right Side
    c_v2_unit = [None] * R2
    dist_u = nus2[-1] - nus2[-2]
    c_v2_unit[-1] = ((torch.exp(dist_u/sigs2[-1]), -torch.exp(-dist_u/sigs2[-1])))

    # recursively calculating c^2
    for j in range(R2 - 1, 0, -1):
        P = c_v2_unit[j][0] + c_v2_unit[j][1]
        S = (1/sigs2[j]) * (-c_v2_unit[j][0] + c_v2_unit[j][1])
        dist = nus2[j] - nus2[j-1]
        c_v2_unit[j-1] = (0.5 * (P - sigs2[j-1]*S) * torch.exp(dist/sigs2[j-1]),
                          0.5 * (P + sigs2[j-1]*S) * torch.exp(-dist/sigs2[j-1]))

    # Unscaled left-side value at S0
    v1 = c_v1_unit[-1][0] + c_v1_unit[-1][1]

    # Unscaled right-side value at S0
    v2 = c_v2_unit[0][0] + c_v2_unit[0][1]

    # Unscaled left-side derivative at S0
    Dk_v1 = (1/sigs1[-1]) * (-c_v1_unit[-1][0] + c_v1_unit[-1][1])

    # Unscaled right-side derivative at S0
    Dk_v2 = (1/sigs2[0]) * (-c_v2_unit[0][0] + c_v2_unit[0][1])

    # Values for lambda1 and lambda2
    lam1 = v2 / (Dk_v1 * v2 - v1 * Dk_v2)
    lam2 = v1 / (Dk_v1 * v2 - v1 * Dk_v2)

    c_v1 = [(c1 * lam1, c2 * lam1) for (c1, c2) in c_v1_unit]
    c_v2 = [(c1 * lam2, c2 * lam2) for (c1, c2) in c_v2_unit]

    jumps = []

    # Left Side Jumps
    for j in range(R1 - 1):
        v_left = (1/sigs1[j]**2) * (c_v1[j][0] + c_v1[j][1])
        dist = nus1[j+2] - nus1[j+1]
        v_right = (1/sigs1[j+1]**2) * (c_v1[j+1][0]*torch.exp(dist/sigs1[j+1]) +
                                             c_v1[j+1][1]*torch.exp(-dist/sigs1[j+1]))
        jumps.append(v_right - v_left)

    # Jump at S0
    v1_S0 = (1/sigs1[-1]**2) * (c_v1[-1][0] + c_v1[-1][1])
    v2_S0 = (1/sigs2[0]**2) * (c_v2[0][0] + c_v2[0][1])

    jumps.append(v2_S0 - v1_S0)

    # Right Side Jumps
    for j in range(R2 - 1):
        v_right = (1/sigs2[j+1]**2) * (c_v2[j+1][0] + c_v2[j+1][1])
        dist = nus2[j+1] - nus2[j]
        v_left = (1/sigs2[j]**2) * (c_v2[j][0]*torch.exp(-dist/sigs2[j]) +
                                             c_v2[j][1]*torch.exp(dist/sigs2[j]))
        jumps.append(v_right - v_left)

    j = torch.sum(torch.stack(jumps) ** 2)
    return j, lam1, lam2

# Tests

## Test 1

In [ ]:
R1, R2 = 4, 3
S0 = 1271.87

initial_sigs1 = torch.tensor([149.0, 130.0, 250.0, 160.0])
initial_sigs2 = torch.tensor([155.0, 180.0, 210.0])
sigs1_raw = torch.log(torch.exp(initial_sigs1) - 1)
sigs2_raw = torch.log(torch.exp(initial_sigs2) - 1)

theta_old = torch.cat([
  torch.linspace(800, 1200, R1),
  sigs1_raw,
  torch.linspace(1400, 2200, R2),
  sigs2_raw
]).clone().detach().requires_grad_(True)

theta_initial = theta_old.clone().detach()

J_old, lam1, lam2 = calculate_J(
  R1, R2, S0, theta_old
)
J_old.backward()
old_gradients = theta_old.grad.clone().detach()

print(f"Current Smoothness J_old: {J_old.item():.12f}")
print(f"Calculated Lambda 1:  {lam1.item():.12f}")
print(f"Calculated Lambda 2: {lam2.item():.12f}\n")

Current Smoothness J_old: 0.000003114399
Calculated Lambda 1:  4.033944160692
Calculated Lambda 2: 0.672975789580



# Line Search using torch.optim.LBFGS

## Test 1

In [ ]:
# class torch.optim.LBFGS(params, lr=1, max_iter=20, max_eval=None, tolerance_grad=1e-07, tolerance_change=1e-09, history_size=100, line_search_fn=None)
# https://docs.pytorch.org/docs/2.12/generated/torch.optim.LBFGS.html#torch.optim.LBFGS.step$0

# line search
line_search = torch.optim.LBFGS([theta_old], max_iter=50, tolerance_grad=-1e-15, tolerance_change=-1e-15, line_search_fn="strong_wolfe")

def closure():
  line_search.zero_grad()
  J, _, _ = calculate_J(R1, R2, S0, theta_old)
  # J value too small, need to scale
  scaled_J = J * 1e7
  scaled_J.backward()
  return scaled_J

line_search.step(closure)

J_optimized, _, _ = calculate_J(R1, R2, S0, theta_old)

# The following print statements are vibe-coded
print(f"Optimized J Value: {J_optimized.item():.12f}\n")

labels = []
labels.extend([f"nu1_{i}" for i in range(R1)])
labels.extend([f"sig1_raw_{i}" for i in range(R1)])
labels.extend([f"nu2_{i}" for i in range(R2)])
labels.extend([f"sig2_raw_{i}" for i in range(R2)])

print(f"{'Parameter':<15} | {'Old Value':<15} | {'Optimized Value':<15}")
print("-" * 53)

for i in range(len(theta_old)):
    print(f"{labels[i]:<15} | "
          f"{theta_initial[i].item():<15.6f} | "
          f"{theta_old[i].item():<15.6f}")

print(f"Total Penalty Reduction: {((J_old - J_optimized)/J_old * 100).item():.12f}%\n")


Optimized J Value: 0.000000000000

Parameter       | Old Value       | Optimized Value
-----------------------------------------------------
nu1_0           | 800.000000      | 804.015937     
nu1_1           | 933.333333      | 927.168334     
nu1_2           | 1066.666667     | 1056.015443    
nu1_3           | 1200.000000     | 1172.581942    
sig1_raw_0      | 149.000000      | 184.602876     
sig1_raw_1      | 130.000000      | 184.602876     
sig1_raw_2      | 250.000000      | 184.602876     
sig1_raw_3      | 160.000000      | 184.602876     
nu2_0           | 1400.000000     | 1406.797316    
nu2_1           | 1800.000000     | 1803.707728    
nu2_2           | 2200.000000     | 2199.869664    
sig2_raw_0      | 155.000000      | 184.602876     
sig2_raw_1      | 180.000000      | 184.602876     
sig2_raw_2      | 210.000000      | 184.602876     
Total Penalty Reduction: 100.000000000000%



## Test 2

In [ ]:
R1, R2 = 20, 20
S0 = 1400

initial_sigs1 = torch.randint(100, 300, (20,))
initial_sigs2 = torch.randint(100, 300, (20,))
# initial_sigs1 = torch.randint(400, 600, (20,))
# initial_sigs2 = torch.randint(400, 600, (20,))
sigs1_raw = torch.log(torch.exp(initial_sigs1) - 1)
sigs2_raw = torch.log(torch.exp(initial_sigs2) - 1)

theta_old = torch.cat([
  torch.linspace(700, 1350, R1),
  sigs1_raw,
  torch.linspace(1450, 2300, R2),
  sigs2_raw
]).clone().detach().requires_grad_(True)

theta_initial = theta_old.clone().detach()

J_old, lam1, lam2 = calculate_J(
  R1, R2, S0, theta_old
)
J_old.backward()
old_gradients = theta_old.grad.clone().detach()

print(f"Current Smoothness J_old: {J_old.item():.12f}")
print(f"Calculated Lambda 1:  {lam1.item():.12f}")
print(f"Calculated Lambda 2: {lam2.item():.12f}\n")

Current Smoothness J_old: 0.000012179321
Calculated Lambda 1:  2.818358810890
Calculated Lambda 2: 0.567630190227



In [ ]:
line_search = torch.optim.LBFGS([theta_old], max_iter=50, tolerance_grad=-1e-15, tolerance_change=-1e-15, line_search_fn="strong_wolfe")

line_search.step(closure)

J_optimized, _, _ = calculate_J(R1, R2, S0, theta_old)

# The following print statements are vibe-coded
print(f"Optimized J Value: {J_optimized.item():.12f}\n")

labels = []
labels.extend([f"nu1_{i}" for i in range(R1)])
labels.extend([f"sig1_raw_{i}" for i in range(R1)])
labels.extend([f"nu2_{i}" for i in range(R2)])
labels.extend([f"sig2_raw_{i}" for i in range(R2)])

print(f"{'Parameter':<15} | {'Old Value':<15} | {'Optimized Value':<15}")
print("-" * 53)

for i in range(len(theta_old)):
    print(f"{labels[i]:<15} | "
          f"{theta_initial[i].item():<15.6f} | "
          f"{theta_old[i].item():<15.6f}")

print(f"Total Penalty Reduction: {((J_old - J_optimized)/J_old * 100).item():.12f}%\n")


Optimized J Value: 0.000000000299

Parameter       | Old Value       | Optimized Value
-----------------------------------------------------
nu1_0           | 700.000000      | 733.388823     
nu1_1           | 734.210526      | 730.809645     
nu1_2           | 768.421053      | 767.837094     
nu1_3           | 802.631579      | 801.271416     
nu1_4           | 836.842105      | 813.937082     
nu1_5           | 871.052632      | 843.839245     
nu1_6           | 905.263158      | 899.591311     
nu1_7           | 939.473684      | 938.422068     
nu1_8           | 973.684211      | 965.145788     
nu1_9           | 1007.894737     | 1008.716241    
nu1_10          | 1042.105263     | 1045.517377    
nu1_11          | 1076.315789     | 1035.830562    
nu1_12          | 1110.526316     | 1119.775349    
nu1_13          | 1144.736842     | 1120.486192    
nu1_14          | 1178.947368     | 1185.901022    
nu1_15          | 1213.157895     | 1203.328532    
nu1_16          | 1247.3684

## Test 3

In [ ]:
R1, R2 = 70, 50
S0 = 1730

initial_sigs1 = torch.randint(500, 700, (70,))
initial_sigs2 = torch.randint(500, 700, (50,))
sigs1_raw = torch.log(torch.exp(initial_sigs1) - 1)
sigs2_raw = torch.log(torch.exp(initial_sigs2) - 1)

theta_old = torch.cat([
  torch.linspace(600, 1700, R1),
  sigs1_raw,
  torch.linspace(1750, 3120, R2),
  sigs2_raw
]).clone().detach().requires_grad_(True)

theta_initial = theta_old.clone().detach()

J_old, lam1, lam2 = calculate_J(
  R1, R2, S0, theta_old
)
J_old.backward()
old_gradients = theta_old.grad.clone().detach()

print(f"Current Smoothness J_old: {J_old.item():.12f}")
print(f"Calculated Lambda 1:  {lam1.item():.12f}")
print(f"Calculated Lambda 2: {lam2.item():.12f}\n")

line_search = torch.optim.LBFGS([theta_old], max_iter=50, tolerance_grad=-1e-15, tolerance_change=-1e-15, line_search_fn="strong_wolfe")

line_search.step(closure)

J_optimized, _, _ = calculate_J(R1, R2, S0, theta_old)

# The following print statements are vibe-coded
print(f"Optimized J Value: {J_optimized.item():.12f}\n")

labels = []
labels.extend([f"nu1_{i}" for i in range(R1)])
labels.extend([f"sig1_raw_{i}" for i in range(R1)])
labels.extend([f"nu2_{i}" for i in range(R2)])
labels.extend([f"sig2_raw_{i}" for i in range(R2)])

print(f"{'Parameter':<15} | {'Old Value':<15} | {'Optimized Value':<15}")
print("-" * 53)

for i in range(len(theta_old)):
    print(f"{labels[i]:<15} | "
          f"{theta_initial[i].item():<15.6f} | "
          f"{theta_old[i].item():<15.6f}")

print(f"Total Penalty Reduction: {((J_old - J_optimized)/J_old * 100).item():.12f}%\n")


Current Smoothness J_old: 0.000001118024
Calculated Lambda 1:  48.139202788913
Calculated Lambda 2: 27.812696930237

Optimized J Value: 0.000000000104

Parameter       | Old Value       | Optimized Value
-----------------------------------------------------
nu1_0           | 600.000000      | 688.849537     
nu1_1           | 615.942029      | 616.542256     
nu1_2           | 631.884058      | 676.914565     
nu1_3           | 647.826087      | 672.867942     
nu1_4           | 663.768116      | 690.592395     
nu1_5           | 679.710145      | 686.706090     
nu1_6           | 695.652174      | 695.603886     
nu1_7           | 711.594203      | 708.714200     
nu1_8           | 727.536232      | 726.962810     
nu1_9           | 743.478261      | 729.154931     
nu1_10          | 759.420290      | 757.354731     
nu1_11          | 775.362319      | 774.415082     
nu1_12          | 791.304348      | 783.721673     
nu1_13          | 807.246377      | 762.605401     
nu1_14        

# Bisection Line Search

In [ ]:
# https://fiveable.me/mathematical-methods-for-optimization/unit-8/line-search-methods/study-guide/ntbjNVC8WK1LTZsf$0
# https://www.cs.cmu.edu/~pradeepr/convexopt/Lecture_Slides/Descent-Line-Search.pdf

def bisection_line_search(theta, gradient, loss_fn, R1, R2, S0, alpha_max=10, tol=1e-5, max_iter=20):

    alpha_left = 0
    alpha_right = alpha_max

    best_theta = theta.clone().detach()
    best_loss = float('inf')
    best_alpha = 0

    for iteration in range(max_iter):
        alpha_mid = (alpha_left + alpha_right)/2

        new_theta = theta - alpha_mid * gradient
        new_param = new_theta.clone().detach().requires_grad_(True)

        J_candidate, _, _ = loss_fn(R1, R2, S0, new_param)
        scaled_J = J_candidate * 1e7
        scaled_J.backward()

        # p = grad(theta_mid) (dot product) (initial_gradient)
        curr_grad = new_param.grad.clone().detach()
        p = torch.sum(- curr_grad * gradient).item()

        if J_candidate.item() < best_loss:
            best_loss = J_candidate.item()
            best_theta = new_theta.clone().detach()
            best_alpha = alpha_mid

        if (alpha_right - alpha_left) < tol:
            break

        if p > 0:
            # > min, move right bound inward
            alpha_right = alpha_mid
        else:
            alpha_left = alpha_mid

    return best_theta, best_alpha, best_loss

## Test 1

In [ ]:
R1, R2 = 4, 3
S0 = 1271.87

initial_sigs1 = torch.tensor([149.0, 130.0, 250.0, 160.0])
initial_sigs2 = torch.tensor([155.0, 180.0, 210.0])
sigs1_raw = torch.log(torch.exp(initial_sigs1) - 1)
sigs2_raw = torch.log(torch.exp(initial_sigs2) - 1)

theta = torch.cat([
  torch.linspace(800, 1200, R1),
  sigs1_raw,
  torch.linspace(1400, 2200, R2),
  sigs2_raw
]).clone().detach().requires_grad_(True)

for i in range(30):
    J, _, _ = calculate_J(R1, R2, S0, theta)
    scaled_J = J * 1e7
    scaled_J.backward()

    init_gradient = theta.grad.clone().detach()
    theta.grad.zero_()

    theta_new, alpha, curr_loss = bisection_line_search(theta=theta, gradient=init_gradient, loss_fn=calculate_J, R1=R1, R2=R2, S0=S0,
                                                        alpha_max=20, tol=1e-6, max_iter=15)

    print(f"Iteration {i:02d} | Alpha: {alpha:<9.6f} | Current J: {curr_loss:.12f}")

    with torch.no_grad():
        theta.copy_(theta_new)
        idx = 0
        nus1 = theta[idx : idx + R1]; idx += R1
        sigs1_raw_curr = theta[idx : idx + R1]; idx += R1
        nus2 = theta[idx : idx + R2]; idx += R2
        sigs2_raw_curr = theta[idx : idx + R2]; idx += R2

        sigs1 = torch.nn.functional.softplus(sigs1_raw_curr) + 1e-6
        sigs2 = torch.nn.functional.softplus(sigs2_raw_curr) + 1e-6

    print(f"  Left Side:")
    for i in range(R1):
        print(f"   nu{i+1} = {nus1[i].item():11.4f} | sigma{i+1} = {sigs1[i].item():11.4f}")
    print(f"  Right Side:")
    for i in range(R2):
        print(f"   nu{i+1} = {nus2[i].item():11.4f} | sigma{i+1} = {sigs2[i].item():11.4f}")
    print("-" * 60)

Iteration 00 | Alpha: 19.999390 | Current J: 0.000002206429
  Left Side:
   nu1 =    800.2467 | sigma1 =    148.6308
   nu2 =    933.2022 | sigma2 =    137.7496
   nu3 =   1065.1296 | sigma3 =    243.5367
   nu4 =   1194.1405 | sigma4 =    167.2673
  Right Side:
   nu1 =   1400.6752 | sigma1 =    158.4918
   nu2 =   1800.0059 | sigma2 =    176.9797
   nu3 =   2199.9992 | sigma3 =    209.9783
------------------------------------------------------------
Iteration 01 | Alpha: 19.999390 | Current J: 0.000001640944
  Left Side:
   nu1 =    800.4412 | sigma1 =    148.3809
   nu2 =    933.1528 | sigma2 =    143.6800
   nu3 =   1063.8757 | sigma3 =    237.9435
   nu4 =   1190.4654 | sigma4 =    169.9184
  Right Side:
   nu1 =   1401.0403 | sigma1 =    164.2998
   nu2 =   1800.0119 | sigma2 =    174.6397
   nu3 =   2199.9985 | sigma3 =    209.9571
------------------------------------------------------------
Iteration 02 | Alpha: 19.999390 | Current J: 0.000001259971
  Left Side:
   nu1 =    800

In [ ]:
def bisection_line_search_test(R1, R2, S0, theta, iter, max_iter, alpha_max):
  for i in range(iter):
    J, _, _ = calculate_J(R1, R2, S0, theta)
    scaled_J = J * 1e7
    scaled_J.backward()

    init_gradient = theta.grad.clone().detach()
    theta.grad.zero_()

    theta_new, alpha, curr_loss = bisection_line_search(theta=theta, gradient=init_gradient, loss_fn=calculate_J, R1=R1, R2=R2, S0=S0,
                                                        alpha_max=alpha_max, tol=1e-6, max_iter=max_iter)

    print(f"Iteration {i:02d} | Alpha: {alpha:<9.6f} | Current J: {curr_loss:.12f}")

    with torch.no_grad():
        theta.copy_(theta_new)
        idx = 0
        nus1 = theta[idx : idx + R1]; idx += R1
        sigs1_raw_curr = theta[idx : idx + R1]; idx += R1
        nus2 = theta[idx : idx + R2]; idx += R2
        sigs2_raw_curr = theta[idx : idx + R2]; idx += R2

        sigs1 = torch.nn.functional.softplus(sigs1_raw_curr) + 1e-6
        sigs2 = torch.nn.functional.softplus(sigs2_raw_curr) + 1e-6

    print(f"  Left Side:")
    for i in range(R1):
        print(f"   nu{i+1} = {nus1[i].item():11.4f} | sigma{i+1} = {sigs1[i].item():11.4f}")

    print(f"  Right Side:")
    for i in range(R2):
        print(f"   nu{i+1} = {nus2[i].item():11.4f} | sigma{i+1} = {sigs2[i].item():11.4f}")

    print("-" * 60)

## Test 2

In [ ]:
R1, R2 = 20, 20
S0 = 1400

initial_sigs1 = torch.randint(100, 300, (20,))
initial_sigs2 = torch.randint(100, 300, (20,))
# initial_sigs1 = torch.randint(400, 600, (20,))
# initial_sigs2 = torch.randint(400, 600, (20,))
sigs1_raw = torch.log(torch.exp(initial_sigs1) - 1)
sigs2_raw = torch.log(torch.exp(initial_sigs2) - 1)

theta = torch.cat([
  torch.linspace(700, 1350, R1),
  sigs1_raw,
  torch.linspace(1450, 2300, R2),
  sigs2_raw
]).clone().detach().requires_grad_(True)

bisection_line_search_test(R1, R2, S0, theta, 30, 30, 20)

# for i in range(30):
#     J, _, _ = calculate_J(R1, R2, S0, theta)
#     scaled_J = J * 1e7
#     scaled_J.backward()

#     init_gradient = theta.grad.clone().detach()
#     theta.grad.zero_()

#     theta_new, alpha, curr_loss = bisection_line_search(theta=theta, gradient=init_gradient, loss_fn=calculate_J, R1=R1, R2=R2, S0=S0,
#                                                         alpha_max=20, tol=1e-6, max_iter=15)

#     print(f"Iteration {i:02d} | Alpha: {alpha:<9.6f} | Current J: {curr_loss:.12f}")

#     with torch.no_grad():
#         theta.copy_(theta_new)
#         idx = 0
#         nus1 = theta[idx : idx + R1]; idx += R1
#         sigs1_raw_curr = theta[idx : idx + R1]; idx += R1
#         nus2 = theta[idx : idx + R2]; idx += R2
#         sigs2_raw_curr = theta[idx : idx + R2]; idx += R2

#         sigs1 = torch.nn.functional.softplus(sigs1_raw_curr) + 1e-6
#         sigs2 = torch.nn.functional.softplus(sigs2_raw_curr) + 1e-6

#     print(f"  Left Side:")
#     for i in range(R1):
#         print(f"   nu{i+1} = {nus1[i].item():11.4f} | sigma{i+1} = {sigs1[i].item():11.4f}")

#     print(f"  Right Side:")
#     for i in range(R2):
#         print(f"   nu{i+1} = {nus2[i].item():11.4f} | sigma{i+1} = {sigs2[i].item():11.4f}")

#     print("-" * 60)

Iteration 00 | Alpha: 3.856200  | Current J: 0.000013870187
  Left Side:
   nu1 =    700.0359 | sigma1 =    271.0000
   nu2 =    734.2105 | sigma2 =    264.9996
   nu3 =    768.4193 | sigma3 =    204.9984
   nu4 =    802.6363 | sigma4 =    297.9972
   nu5 =    836.7951 | sigma5 =    124.0054
   nu6 =    871.0981 | sigma6 =    210.9812
   nu7 =    905.2354 | sigma7 =    166.9810
   nu8 =    939.5349 | sigma8 =    261.9634
   nu9 =    973.2318 | sigma9 =    102.4813
   nu10 =   1008.2577 | sigma10 =    258.9351
   nu11 =   1042.0467 | sigma11 =    215.9506
   nu12 =   1076.4040 | sigma12 =    270.9473
   nu13 =   1110.5248 | sigma13 =    269.9145
   nu14 =   1144.4326 | sigma14 =    188.6716
   nu15 =   1178.2518 | sigma15 =    139.3909
   nu16 =   1214.0714 | sigma16 =    214.9642
   nu17 =   1243.8104 | sigma17 =    120.5127
   nu18 =   1283.5699 | sigma18 =    201.2852
   nu19 =   1310.9451 | sigma19 =    126.8862
   nu20 =   1351.0691 | sigma20 =    120.8113
  Right Side:
   nu1 =   

## Test 3

In [ ]:
R1, R2 = 70, 50
S0 = 1730

initial_sigs1 = torch.randint(500, 700, (70,))
initial_sigs2 = torch.randint(500, 700, (50,))
sigs1_raw = torch.log(torch.exp(initial_sigs1) - 1)
sigs2_raw = torch.log(torch.exp(initial_sigs2) - 1)

theta = torch.cat([
  torch.linspace(600, 1700, R1),
  sigs1_raw,
  torch.linspace(1750, 3120, R2),
  sigs2_raw
]).clone().detach().requires_grad_(True)

bisection_line_search_test(R1, R2, S0, theta, 30, 20, 10)

Iteration 00 | Alpha: 9.999990  | Current J: 0.000001078550
  Left Side:
   nu1 =    600.0456 | sigma1 =    578.0000
   nu2 =    615.9420 | sigma2 =    683.0000
   nu3 =    631.8837 | sigma3 =    525.0001
   nu4 =    647.8261 | sigma4 =    563.9999
   nu5 =    663.7681 | sigma5 =    551.0001
   nu6 =    679.7102 | sigma6 =    574.0001
   nu7 =    695.6523 | sigma7 =    640.9995
   nu8 =    711.5937 | sigma8 =    574.0002
   nu9 =    727.5361 | sigma9 =    564.0009
   nu10 =    743.4786 | sigma10 =    671.9992
   nu11 =    759.4201 | sigma11 =    646.9997
   nu12 =    775.3618 | sigma12 =    606.0001
   nu13 =    791.3043 | sigma13 =    602.9979
   nu14 =    807.2432 | sigma14 =    513.0062
   nu15 =    823.1897 | sigma15 =    609.9978
   nu16 =    839.1308 | sigma16 =    626.9977
   nu17 =    855.0698 | sigma17 =    557.0050
   nu18 =    871.0165 | sigma18 =    647.9984
   nu19 =    886.9577 | sigma19 =    697.9949
   nu20 =    902.8880 | sigma20 =    526.0120
   nu21 =    918.8428 | s